# Student Performance Analysis and Prediction System

**Minor Project | AI/ML with Python Training Program**

This notebook uses the UCI Student Performance Portuguese-course dataset (649 records). Studytime is a weekly category, absences are a count, and G1/G2 are earlier-period grades. The target is G3 (0-20).

## 1. Import libraries and inspect the dataset

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from model import prepare_data, make_eda, train_evaluate

df_raw = pd.read_csv(ROOT / 'data' / 'student_performance_dataset.csv')
print('Shape:', df_raw.shape)
display(df_raw.head())
print('\nMissing values:')
display(df_raw.isna().sum())
print('\nData types:')
display(df_raw.dtypes)

## 2. Data cleaning and feature engineering

The five-column working file has repeated combinations because different students can share the same selected values. Those combinations are retained. The full original UCI source was separately checked for exact duplicate rows.

In [ ]:
df, cleaning_summary = prepare_data()
print(json.dumps(cleaning_summary, indent=2))
display(df.head())

## 3. Descriptive statistics

In [ ]:
numeric = df[['studytime', 'absences', 'G1', 'G2', 'G3']]
stats = numeric.describe().T
stats['median'] = numeric.median()
display(stats.round(3))

## 4. Exploratory data analysis and required charts

In [ ]:
eda = make_eda(df)
print('Mean, median and standard deviation of G3:')
print(eda['mean_G3'], eda['median_G3'], eda['std_G3'])
print('Correlations with G3:')
display(pd.Series(eda['correlations_with_G3'], name='correlation'))

plot_dir = ROOT / 'outputs' / 'visualizations'
for name in ['bar_mean_grade_by_studytime.png', 'histogram_final_grade.png', 'scatter_studytime_vs_final_grade.png', 'scatter_absences_vs_final_grade.png', 'scatter_g2_vs_final_grade.png', 'boxplot_grade_by_studytime.png', 'correlation_heatmap.png']:
    display(plt.imread(plot_dir / name).shape)
    plt.figure(figsize=(8, 5))
    plt.imshow(plt.imread(plot_dir / name))
    plt.axis('off')
    plt.title(name)
    plt.show()

## 5. Linear Regression model training and evaluation

An 80/20 split is used. Imputation and scaling are part of the training pipeline, so they are fitted within each training fold. GridSearchCV compares the intercept setting using five-fold cross-validation.

In [ ]:
metrics = train_evaluate(df)
print(json.dumps(metrics, indent=2))

## 6. Predictions and conclusion

In [ ]:
predictions = pd.read_csv(ROOT / 'outputs' / 'evaluation_results' / 'test_predictions.csv')
display(predictions.head(15).round(2))
print('MAE:', round(metrics['test_MAE'], 3))
print('RMSE:', round(metrics['test_RMSE'], 3))
print('R2:', round(metrics['test_R2'], 3))

G2 and G1 are the strongest linear correlates of G3 in this dataset. Studytime has a smaller positive association, while absences have a weak negative association. These are observational relationships, not evidence of causation. Because G1 and G2 are previous-period marks, the system should only be used once they are known. Studytime is categorical and absences are not an attendance percentage.